# Landing Zone — Alpha Vantage Stock Fetcher

Fetches daily OHLCV data for each symbol in `WATCHLIST` from the Alpha Vantage API
and saves one CSV per symbol to `/home/landing/stocks/daily/`.

**Parameters (Cell 2):**
- `WATCHLIST` — list of ticker symbols to fetch
- `TARGET_DATE` — `None` for the latest available trading day, or `"YYYY-MM-DD"` for a specific date

**No Spark required** — pure Python, independent of the Spark cluster.

Free tier limits: **25 calls/day**, **5 calls/minute**

In [ ]:
WATCHLIST   = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]
TARGET_DATE = None   # None = latest trading day  |  "YYYY-MM-DD" = specific date

## Setup

In [ ]:
import os
from datetime import date, datetime

ALPHAVANTAGE_API_KEY = os.environ.get("ALPHAVANTAGE_API_KEY", "")
OUTPUT_DIR = "/home/landing/stocks/daily"
BASE_URL   = "https://www.alphavantage.co/query"

if not ALPHAVANTAGE_API_KEY:
    raise EnvironmentError(
        "ALPHAVANTAGE_API_KEY is not set. "
        "Add it to your .env file and restart the jupyter container."
    )

# Validate TARGET_DATE and determine API outputsize
if TARGET_DATE is None:
    OUTPUT_SIZE  = "compact"
    display_date = "latest trading day"
else:
    try:
        target = datetime.strptime(TARGET_DATE, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"TARGET_DATE must be None or 'YYYY-MM-DD', got: {TARGET_DATE!r}")
    if target >= date.today():
        raise ValueError(f"TARGET_DATE must be a past date, got: {TARGET_DATE}")
    # Use compact (last ~100 trading days) if within ~5 months, else full history
    OUTPUT_SIZE  = "compact" if (date.today() - target).days <= 140 else "full"
    display_date = TARGET_DATE

print(f"Watchlist   : {WATCHLIST}")
print(f"Target date : {display_date}")
print(f"Output size : {OUTPUT_SIZE}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"API key     : {'*' * (len(ALPHAVANTAGE_API_KEY) - 4)}{ALPHAVANTAGE_API_KEY[-4:]}")

## Fetch & Save Functions

In [ ]:
import csv
import requests
from pathlib import Path


def fetch_daily(symbol: str, target_date: str | None, output_size: str, api_key: str) -> dict | None:
    """
    Fetches TIME_SERIES_DAILY from Alpha Vantage.
    - target_date=None  → returns the most recent trading day
    - target_date=str   → filters to that specific date; returns None if not a trading day
    """
    params = {
        "function":   "TIME_SERIES_DAILY",
        "symbol":     symbol,
        "outputsize": output_size,
        "datatype":   "json",
        "apikey":     api_key,
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    ts = data.get("Time Series (Daily)")
    if not ts:
        msg = data.get("Note") or data.get("Information") or str(data)
        print(f"→ API error: {msg}")
        return None

    if target_date is None:
        chosen_date = max(ts.keys())
    else:
        if target_date not in ts:
            nearest = min(ts.keys(), key=lambda d: abs(
                (datetime.strptime(d, "%Y-%m-%d") - datetime.strptime(target_date, "%Y-%m-%d")).days
            ))
            print(f"→ {target_date} is not a trading day. Nearest: {nearest}")
            return None
        chosen_date = target_date

    row = ts[chosen_date]
    return {
        "symbol": symbol,
        "date":   chosen_date,
        "open":   row["1. open"],
        "high":   row["2. high"],
        "low":    row["3. low"],
        "close":  row["4. close"],
        "volume": row["5. volume"],
    }


def save_csv(record: dict, output_dir: str) -> str:
    """Saves a single OHLCV record as a CSV. Returns the path written."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filepath = os.path.join(output_dir, f"{record['symbol']}_{record['date']}.csv")
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["symbol", "date", "open", "high", "low", "close", "volume"])
        writer.writeheader()
        writer.writerow(record)
    return filepath


print("Functions defined.")

## Fetch All Symbols

In [ ]:
import time

RATE_LIMIT_SLEEP = 13  # seconds — keeps under 5 calls/min (free tier)

results = []
skipped = []
errors  = []

print(f"Fetching {len(WATCHLIST)} symbol(s) — {display_date}...\n")

for i, symbol in enumerate(WATCHLIST):
    print(f"[{i+1}/{len(WATCHLIST)}] {symbol} ", end="", flush=True)
    try:
        record = fetch_daily(symbol, TARGET_DATE, OUTPUT_SIZE, ALPHAVANTAGE_API_KEY)
        if record:
            path = save_csv(record, OUTPUT_DIR)
            results.append({"symbol": symbol, "date": record["date"], "close": record["close"], "file": path})
            print(f"→ {record['date']}  close={record['close']}  saved.")
        else:
            skipped.append(symbol)
    except Exception as exc:
        print(f"→ ERROR: {exc}")
        errors.append(symbol)

    if i < len(WATCHLIST) - 1:
        print(f"   (waiting {RATE_LIMIT_SLEEP}s...)")
        time.sleep(RATE_LIMIT_SLEEP)

print(f"\nDone. {len(results)} saved, {len(skipped)} skipped, {len(errors)} errors.")

## Summary

In [ ]:
import pandas as pd

if results:
    df = pd.DataFrame(results)
    print(f"Fetched {len(results)}/{len(WATCHLIST)} symbols — {df['date'].iloc[0]}")
    print(f"Files written to: {OUTPUT_DIR}\n")
    display(df)
else:
    print("No data saved. Check the messages above for details.")